# GraphRAG demo: Neo4j + LangChain + Gemini

**What we will do**
1. Write two small text files (our "documents")
2. Load them and split them into chunks
3. Ask Gemini to extract **entities** (nodes) and **relationships**
4. Store them in **Neo4j** (LangChain writes the Cypher `MERGE` for us)
5. Ask questions in plain English → Gemini writes a **Cypher** query → Neo4j answers

**Before you start**
- Neo4j running locally (Neo4j Desktop or Docker) at `bolt://localhost:7687`
- In Neo4j Desktop, install the **APOC** plugin for your database (LangChain uses it to read the graph schema)
- A Google AI Studio API key: https://aistudio.google.com/apikey

## Step 0 · Install packages

In [1]:
%pip install -q langchain langchain-community langchain-neo4j langchain-experimental langchain-google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Step 1 · Settings

In [16]:
import os
from dotenv import load_dotenv

# Read settings from the .env file in this folder
load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")   # langchain-google-genai reads this automatically
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")

assert GOOGLE_API_KEY, "GOOGLE_API_KEY is missing from .env"
assert NEO4J_PASSWORD, "NEO4J_PASSWORD is missing from .env"
print("Settings loaded:", NEO4J_URI, NEO4J_USERNAME, GEMINI_MODEL)

Settings loaded: neo4j://127.0.0.1:7687 neo4j gemini-3.5-flash-lite


## Step 2 · Create two small input files

In [17]:
os.makedirs("data", exist_ok=True)

doc1 = """Alice is a data engineer at Neo4j. Alice lives in London.
Bob is a machine learning engineer at Google. Bob lives in Bengaluru.
Alice and Bob both studied at IIT Hyderabad."""

doc2 = """Neo4j has an office in London.
Google has offices in Bengaluru and Hyderabad.
Alice works on the GraphRAG project. Bob works on the Gemini project.
The GraphRAG project uses Gemini."""

with open("data/team.txt", "w") as f:
    f.write(doc1)
with open("data/projects.txt", "w") as f:
    f.write(doc2)

print("Created data/team.txt and data/projects.txt")

Created data/team.txt and data/projects.txt


## Step 3 · Load the files and split them into chunks

In [18]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = TextLoader("data/team.txt").load() + TextLoader("data/projects.txt").load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} documents -> {len(chunks)} chunks")
for c in chunks:
    print("-", c.page_content[:80].replace("\n", " "), "...")

2 documents -> 2 chunks
- Alice is a data engineer at Neo4j. Alice lives in London. Bob is a machine learn ...
- Neo4j has an office in London. Google has offices in Bengaluru and Hyderabad. Al ...


## Step 4 · Connect to Neo4j and the Gemini LLM

In [19]:
from langchain_neo4j import Neo4jGraph
from langchain_google_genai import ChatGoogleGenerativeAI

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)

# Start with an empty database so the demo is repeatable
graph.query("MATCH (n) DETACH DELETE n")
print("Connected to Neo4j and cleared old data")

Connected to Neo4j and cleared old data


## Step 5 · LLM extracts entities and relationships

`LLMGraphTransformer` sends each chunk to Gemini. Gemini returns structured data
(nodes + relationships), and LangChain turns it into **GraphDocument** objects.
No Cypher yet. We also tell it which node and relationship types are allowed, so the graph stays clean.

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

transformer = LLMGraphTransformer(
    llm=llm,
    #allowed_nodes=["Person", "Company", "City", "University", "Project"],
    #allowed_relationships=["WORKS_AT", "LIVES_IN", "STUDIED_AT",
                           #"HAS_OFFICE_IN", "WORKS_ON", "USES"],
    #node_properties=["role"],            # details about a thing
    #relationship_properties=["since"],   # details about a connection
)

graph_docs = transformer.convert_to_graph_documents(chunks)

# Look at what the LLM extracted
for gd in graph_docs:
    print("NODES:")
    for n in gd.nodes:
        print(f"   {n.id} ({n.type})")
    print("RELATIONSHIPS:")
    for r in gd.relationships:
        print(f"   {r.source.id} -[{r.type}]-> {r.target.id}")
    print()

c:\Repos\agenticAI\GraphRAG\.venv-1\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Repos\agenticAI\GraphRAG\.venv-1\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


NODES:
   Alice (Person)
   Neo4J (Organization)
   London (Location)
   Bob (Person)
   Google (Organization)
   Bengaluru (Location)
   Iit Hyderabad (Organization)
RELATIONSHIPS:
   Alice -[EMPLOYEE]-> Neo4J
   Alice -[RESIDENCE]-> London
   Bob -[EMPLOYEE]-> Google
   Bob -[RESIDENCE]-> Bengaluru
   Alice -[STUDENT]-> Iit Hyderabad
   Bob -[STUDENT]-> Iit Hyderabad

NODES:
   Neo4J (Company)
   London (City)
   Google (Company)
   Bengaluru (City)
   Hyderabad (City)
   Alice (Person)
   Graphrag (Project)
   Bob (Person)
   Gemini (Project)
RELATIONSHIPS:
   Neo4J -[LOCATED_IN]-> London
   Google -[LOCATED_IN]-> Bengaluru
   Google -[LOCATED_IN]-> Hyderabad
   Alice -[WORKS_ON]-> Graphrag
   Bob -[WORKS_ON]-> Gemini
   Graphrag -[USES]-> Gemini



## Step 6 · Store the graph in Neo4j

`add_graph_documents` turns the GraphDocuments into Cypher `MERGE` statements and runs them.
`include_source=True` also saves each chunk as a `Document` node linked with `MENTIONS`.

In [21]:
graph.add_graph_documents(graph_docs, include_source=True, baseEntityLabel=True)

graph.refresh_schema()
print(graph.schema)

Node properties:
Document {id: STRING, text: STRING, source: STRING}
Person {id: STRING}
Organization {id: STRING}
Location {id: STRING}
Project {id: STRING}
Company {id: STRING}
City {id: STRING}
Relationship properties:

The relationships:
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:Organization)
(:Document)-[:MENTIONS]->(:Company)
(:Document)-[:MENTIONS]->(:Location)
(:Document)-[:MENTIONS]->(:City)
(:Document)-[:MENTIONS]->(:Project)
(:Person)-[:RESIDENCE]->(:Location)
(:Person)-[:RESIDENCE]->(:City)
(:Person)-[:STUDENT]->(:Organization)
(:Person)-[:WORKS_ON]->(:Project)
(:Person)-[:EMPLOYEE]->(:Organization)
(:Person)-[:EMPLOYEE]->(:Company)
(:Organization)-[:LOCATED_IN]->(:Location)
(:Organization)-[:LOCATED_IN]->(:City)
(:Project)-[:USES]->(:Project)
(:Company)-[:LOCATED_IN]->(:Location)
(:Company)-[:LOCATED_IN]->(:City)


### See the graph
Open Neo4j Browser at http://localhost:7474 and run:
```cypher
MATCH (n) RETURN n
```
Or check it from Python with a Cypher query we write ourselves:

In [7]:
rows = graph.query("""
MATCH (a)-[r]->(b)
WHERE NOT a:Document
RETURN a.id AS from, type(r) AS relationship, b.id AS to
""")
for row in rows:
    print(f"{row['from']} -[{row['relationship']}]-> {row['to']}")

Alice -[WORKS_AT]-> Neo4J
Alice -[LIVES_IN]-> London
Alice -[STUDIED_AT]-> Iit Hyderabad
Alice -[WORKS_ON]-> Graphrag Project
Neo4J -[OFFICE_LOCATION]-> London
Bob -[WORKS_AT]-> Google
Bob -[LIVES_IN]-> Bengaluru
Bob -[STUDIED_AT]-> Iit Hyderabad
Bob -[WORKS_ON]-> Gemini Project
Google -[OFFICE_LOCATION]-> Bengaluru
Google -[OFFICE_LOCATION]-> Hyderabad
Graphrag Project -[USES]-> Gemini Project


## Step 7 · Ask questions

`GraphCypherQAChain` does this for every question:
1. Gemini reads the graph schema + your question and **writes a Cypher query**
2. Neo4j runs the query and returns rows
3. Gemini turns the rows into a plain-English answer

`verbose=True` prints the generated Cypher so students can see it.

In [22]:
from langchain_neo4j import GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template="""You write Cypher queries for a Neo4j graph.

Schema:
{schema}

Rules:
- Use only the node labels, relationship types and properties in the schema.
- Entity names are stored in the `id` property.
- Always match names case-insensitively and partially, like:
  WHERE toLower(n.id) CONTAINS toLower('iit hyderabad')
- Return only the Cypher query, no explanation.

Question: {question}
Cypher query:""",
)

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=cypher_prompt,
    verbose=True,                    # show the generated Cypher
    validate_cypher=True,            # fix relationship directions if needed
    allow_dangerous_requests=True,   # required: the LLM runs queries on your DB
)

In [23]:
result = chain.invoke({"query": "Where does Alice work?"})
print("ANSWER:", result["result"])



> Entering new GraphCypherQAChain chain...


c:\Repos\agenticAI\GraphRAG\.venv-1\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Generated Cypher:
MATCH (p:Person)-[:EMPLOYEE]->(o)
WHERE toLower(p.id) CONTAINS toLower('alice')
RETURN o.id AS Workplace
Full Context:
[{'Workplace': 'Neo4J'}]


c:\Repos\agenticAI\GraphRAG\.venv-1\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



> Finished chain.
ANSWER: Alice works at Neo4J.


In [24]:
questions = [
    "Who studied at IIT hyderabad?",
    #"In which cities does Google have offices?",
    #"Which project does Bob work on?",
    #"Which company has an office in the city where Alice lives?",   # multi-hop
]

for q in questions:
    print("\nQUESTION:", q)
    print("ANSWER:", chain.invoke({"query": q})["result"])


QUESTION: Who studied at IIT hyderabad?


> Entering new GraphCypherQAChain chain...


c:\Repos\agenticAI\GraphRAG\.venv-1\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Generated Cypher:
MATCH (p:Person)-[:STUDENT]->(o:Organization)
WHERE toLower(o.id) CONTAINS toLower('iit hyderabad')
RETURN p.id AS Person
Full Context:
[{'Person': 'Alice'}, {'Person': 'Bob'}]


c:\Repos\agenticAI\GraphRAG\.venv-1\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



> Finished chain.
ANSWER: Alice, Bob studied at IIT hyderabad.


In [ ]:
questions = [
    "What is relationship between Alice and Bob?",
]

for q in questions:
    print("\nQUESTION:", q)
    print("ANSWER:", chain.invoke({"query": q})["result"])


QUESTION: What is relationship between Alice and Bob?


> Entering new GraphCypherQAChain chain...


c:\Repos\agenticAI\GraphRAG\.venv-1\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Generated Cypher:
MATCH p = shortestPath((a:Person)-[:RESIDENCE|STUDENT|WORKS_ON|EMPLOYEE*]-(b:Person))
WHERE toLower(a.id) CONTAINS toLower('alice') AND toLower(b.id) CONTAINS toLower('bob')
RETURN p
Full Context:
[{'p': [{'id': 'Alice'}, 'STUDENT', {'id': 'Iit Hyderabad'}, 'STUDENT', {'id': 'Bob'}]}]


c:\Repos\agenticAI\GraphRAG\.venv-1\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



> Finished chain.
ANSWER: Alice and Bob are both students at Iit Hyderabad.


## Try it yourself
- Add a third text file and re-run Steps 2–6
- Ask your own question in the cell above
- Compare: a question that needs **two hops** (like the last one) is where GraphRAG beats plain vector RAG

**If something fails**
- *APOC error* → install the APOC plugin in Neo4j Desktop and restart the database
- *429 / quota error* → free-tier rate limit; wait a minute and re-run the cell
- *Empty answer* → print the Cypher (`verbose=True`) and check the node names match what Step 5 extracted